#   LangChain 활용 에이전트 활용하기 (ReAct) + 라우팅 기반 다국어 RAG 구현하기

### **학습 목표:** ReAct 프레임워크를 활용한 에이전트 시스템을 개발한다

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import warnings
warnings.filterwarnings("ignore")

---

##  **다국어 RAG 시스템 (LangChain 구현)**

- 옵션 1: **다국어 RAG 시스템**은 언어 교차 검색 기능을 통해 다양한 언어의 문서를 처리함

- 옵션 2: **언어 감지**와 **자동번역** 기능이 통합되어 seamless한 다국어 처리가 가능함

- 옵션 3: **벡터저장소 라우팅**을 통해 각 언어별 최적화된 처리 경로를 구성할 수 있음


---

### 1. **언어 교차(cross-lingual) 검색** 

- **언어 교차 검색**은 서로 다른 언어 간의 정보 검색을 가능하게 하는 기술임

- 질의어와 문서가 **다른 언어**여도 의미적 연관성을 기반으로 검색이 가능함

- **다국어 임베딩**을 활용하여 언어 간 의미적 매칭을 수행함

`(1) 다국어 문서 로드 및 전처리` 

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 데이터 로드
def load_text_files(txt_files):
    data = []

    for text_file in txt_files:
        loader = TextLoader(text_file, encoding='utf-8')
        data += loader.load()

    return data

# 한국어 데이터 로드
korean_txt_files = glob(os.path.join('data', '*_KR.md')) 
korean_data = load_text_files(korean_txt_files)

# 문장을 구분하여 분할 - 정규표현식 사용 (문장 구분자: 마침표, 느낌표, 물음표 다음에 공백이 오는 경우)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",    # TikToken 인코더 이름
    separators=['\n\n', '\n', r'(?<=[.!?])\s+'],   # 구분자
    chunk_size=300,            # 문서 분할 크기
    chunk_overlap=50,          # 문서 분할 중첩  
    is_separator_regex=True,      # 구분자가 정규식인지 여부
    keep_separator=True,          # 구분자 유지 여부
)

korean_docs = text_splitter.split_documents(korean_data)

print("한국어 청크 수:", len(korean_docs))

In [ ]:
# 영어 데이터 로드
english_txt_files = glob(os.path.join('data', '*_EN.md'))
english_data = load_text_files(english_txt_files)

# 문장을 구분하여 분할 - 정규표현식 사용 (문장 구분자: 마침표, 느낌표, 물음표 다음에 공백이 오는 경우)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",    # TikToken 인코더 이름
    separators=['\n\n', '\n', r'(?<=[.!?])\s+'],   # 구분자
    chunk_size=300,            # 문서 분할 크기
    chunk_overlap=50,          # 문서 분할 중첩  
    is_separator_regex=True,      # 구분자가 정규식인지 여부
    keep_separator=True,          # 구분자 유지 여부
)

english_docs = text_splitter.split_documents(english_data)

print("영어 청크 수:", len(english_docs))

`(2) 문서 임베딩 및 벡터저장소에 저장`  

- **3가지 임베딩 모델**(OpenAI, HuggingFace, Ollama)을 활용하여 문서 벡터화 및 성능 비교 
- 한국어 지원: **OpenAI**의 text-embedding-3-small, **HuggingFace**의 bge-m3 모델
- 한국어 미지원: **Ollama**의 nomic-embed-text 모델

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_ollama import OllamaEmbeddings

# OpenAI 임베딩 모델 생성
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")

# Hugoing Face 임베딩 모델 생성
embeddings_huggingface = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# Ollama 임베딩 모델 생성
embeddings_ollama = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
# 다국어 벡터 저장소 구축
from langchain_chroma import Chroma

db_openai = Chroma.from_documents(
    documents=korean_docs+english_docs, 
    embedding=embeddings_openai,
    collection_name="db_openai",
    persist_directory="./chroma_db",
    )

db_huggingface = Chroma.from_documents(
    documents=korean_docs+english_docs, 
    embedding=embeddings_huggingface,
    collection_name="db_huggingface",
    persist_directory="./chroma_db",
    )

db_ollama = Chroma.from_documents(
    documents=korean_docs+english_docs, 
    embedding=embeddings_ollama,
    collection_name="db_ollama",
    persist_directory="./chroma_db",
    )

# 벡터 저장소에 저장된 문서 수
print(f"OpenAI: {db_openai._collection.count()}")
print(f"Hugging Face: {db_huggingface._collection.count()}")
print(f"Ollama: {db_ollama._collection.count()}")

In [ ]:
# 다국어 벡터 저장소 로드
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_ollama import OllamaEmbeddings

# 임베딩 모델 생성
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings_huggingface = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
embeddings_ollama = OllamaEmbeddings(model="nomic-embed-text")

db_openai = Chroma(
    embedding_function=embeddings_openai,
    collection_name="db_openai",
    persist_directory="./chroma_db",
)

db_huggingface = Chroma(
    embedding_function=embeddings_huggingface,
    collection_name="db_huggingface",
    persist_directory="./chroma_db",
)

db_ollama = Chroma(
    embedding_function=embeddings_ollama,
    collection_name="db_ollama",
    persist_directory="./chroma_db",
)

# 벡터 저장소에 저장된 문서 수
print(f"OpenAI: {db_openai._collection.count()}")
print(f"Hugging Face: {db_huggingface._collection.count()}")
print(f"Ollama: {db_ollama._collection.count()}")

`(3) RAG 성능 비교` 

In [ ]:
# RAG 체인 생성
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.chat_models import init_chat_model

# 질문 템플릿 정의 
template = """Answer the question based only on the following context.
Do not use any external information or knowledge.
If the answer is not in the context, answer "I don't know".

When answering:
- For proper nouns (names of people, places, organizations, etc.), provide both Korean and English names in the format: 한글명(English Name) or English Name(한글명)
- Example: 세종대왕(King Sejong), Microsoft(마이크로소프트), New York(뉴욕)

[Context]
{context}

[Question]
{question}

[Answer]
"""

# 프롬프트 생성
prompt = ChatPromptTemplate.from_template(template)

# 문서 포맷터 함수
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

# LLM 모델 생성
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

# 체인 생성
def create_rag_chain(vectorstore):

    retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

    return (
        {"context": retriever | format_docs , "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
# 체인 생성
rag_chain_openai = create_rag_chain(db_openai)
rag_chain_huggingface = create_rag_chain(db_huggingface)
rag_chain_ollama = create_rag_chain(db_ollama)

In [ ]:
# 한국어 쿼리에 대한 성능 평가
query_ko = "테슬라 창업자는 누구인가요?"

# OpenAI
output_openai = rag_chain_openai.invoke(query_ko)
print("OpenAI:", output_openai)

# Hugging Face
output_huggingface = rag_chain_huggingface.invoke(query_ko)
print("Hugging Face:", output_huggingface)

# Ollama
output_ollama = rag_chain_ollama.invoke(query_ko)
print("Ollama:", output_ollama)

In [ ]:
# 영어 쿼리에 대한 성능 평가
query_en = "Who is the founder of Tesla?"

# OpenAI
output_openai = rag_chain_openai.invoke(query_en)
print("OpenAI:", output_openai)

# Hugging Face
output_huggingface = rag_chain_huggingface.invoke(query_en)
print("Hugging Face:", output_huggingface)

# Ollama
output_ollama = rag_chain_ollama.invoke(query_en)
print("Ollama:", output_ollama)

---

### 2. **언어 감지 및 번역 자동화** 

- **langdetect**의 **언어 감지** 기능으로 입력 텍스트의 언어를 자동으로 식별함

- **DeepL**을 통해 감지된 언어로 번역 

- deepl 설치: pip install deepl 또는 uv add deepl (.env 파일에 DEEPL_API_KEY 설정 필요)

- langdetect 설치: pip install langdetect 또는 uv add langdetect

`(1) 한국어 문서 벡터저장소 초기화 ` 

In [ ]:
# 한국어 문서로 저장되어 있는 벡터 저장소 로드

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

print(f"벡터 저장소에 저장된 문서 수: {vectorstore._collection.count()}")

In [ ]:
load_dotenv()

In [ ]:
import deepl
from langdetect import detect

# Deepl 번역기 생성
translator = deepl.Translator(os.getenv('DEEPL_API_KEY'))

# 언어 감지 및 번역 함수
def detect_and_translate(text, target_lang='KO'):
    """ 텍스트의 언어를 감지하고, 목표 언어로 번역합니다."""

    # 언어 감지
    detected_lang = detect(text)

    # 언어가 목표 언어와 다른 경우 번역
    if detected_lang.upper() != target_lang:
        result = translator.translate_text(text, target_lang=target_lang)
        return str(result), detected_lang
    
    # 언어가 목표 언어와 같은 경우 원본 텍스트 반환
    return text, detected_lang

# 문서 번역 테스트 (영어 -> 한국어)
text = "Who is the founder of Tesla?"

translated_text, detected_lang = detect_and_translate(text, target_lang='KO')

print(f"Detected language: {detected_lang}")
print(f"Translated text: {translated_text}")

In [ ]:
# 문서 번역 테스트 (한국어 -> 영어)

text = "테슬라 창업자는 누구인가요?"

translated_text, detected_lang = detect_and_translate(text, target_lang='EN-US')

print(f"Detected language: {detected_lang}")
print(f"Translated text: {translated_text}")

`(2) RAG 체인 성능 평가 `  

In [ ]:
from langchain_core.runnables import chain, RunnablePassthrough

# 벡터저장소 문서를 검색하는 도구 
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# 문서를 검색하여 답변을 생성하는 RAG 체인 생성
lang_rag_chain = (
        {"context": retriever | format_docs , "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

# 언어 감지에 기반한 RAG 실행 함수를 체인으로 변환 (@chain 데코레이터 사용)
@chain
def run_lang_rag_chain(query):

    # 입력 쿼리의 언어 감지
    original_lang = detect(query)
    print(f"Original language: {original_lang}")
    
    # 한국어가 아닌 경우 번역
    if original_lang.upper() != 'KO':
        translated_query, _ = detect_and_translate(query, target_lang='KO')

    # 한국어인 경우 번역 없이 쿼리 사용
    else:
        translated_query = query

    print(f"Translated query: {translated_query}")
    
    # RAG 체인 실행
    output = lang_rag_chain.invoke(translated_query)

    print(f"Output: {output}")
    
    # 번역된 경우 다시 번역 (영어로)
    if original_lang.upper() != 'KO':
        output = translator.translate_text(output, target_lang='EN-US')
    
    return str(output)

In [ ]:
# 한국어 쿼리에 대한 테스트 실행
query_ko = "테슬라 창업자는 누구인가요?"

output = run_lang_rag_chain.invoke(query_ko)
print(output)

In [ ]:
# 영어 쿼리에 대한 테스트 실행 (영어가 섞인 경우 번역 오류 발생 가능)
query_en = "Who is the founder of Tesla?"

output = run_lang_rag_chain.invoke(query_en)
print(output)

---

### 3. **언어 감지 및 벡터저장소 라우팅** 

- **언어별 벡터저장소**를 분리하여 한국어와 영어 문서를 독립적으로 관리함

- 각 언어에 **최적화된 저장소**를 구성하여 검색 효율성을 향상시킴

- **라우팅 시스템**을 통해 언어를 감지하고 해당 벡터저장소로 자동 연결됨

`(1) 언어별 벡터저장소 생성 `  

In [ ]:
# 한국어 문서로 저장되어 있는 벡터 저장소 로드
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

db_korean = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

print(f"한국어 문서 수: {db_korean._collection.count()}")

In [ ]:
# 영어 문서를 저장하는 벡터 저장소 생성
db_english = Chroma.from_documents(
    documents=english_docs, 
    embedding=embeddings_openai,
    collection_name="eng_db_openai",
    persist_directory="./chroma_db",
    )

print(f"영어 문서 수: {db_english._collection.count()}")

In [ ]:
# 영어 문서를 저장하는 벡터 저장소 로드
db_english = Chroma(
    embedding_function=embeddings_openai,
    collection_name="eng_db_openai",
    persist_directory="./chroma_db",
    )

print(f"영어 문서 수: {db_english._collection.count()}")

`(2) RAG 체인 성능 평가 `  

In [ ]:
from langdetect import detect

# 각 언어별 RAG 체인 생성 (한국어, 영어)
rag_chain_korean = create_rag_chain(db_korean)
rag_chain_english = create_rag_chain(db_english)


# 언어 감지에 기반한 RAG 실행 함수를 체인으로 변환 (@chain 데코레이터 사용)
@chain
def run_route_rag_chain(query):

    # 입력 쿼리의 언어 감지
    original_lang = detect(query)
    
    # 한국어인 경우 한국어 RAG 체인 실행 (한국어 문서 벡터 저장소 사용)
    if original_lang.upper() == 'KO':
        return rag_chain_korean.invoke(query)
    
    # 영어인 경우 영어 RAG 체인 실행 (영어 문서 벡터 저장소 사용)
    elif 'EN' in original_lang.upper():
        return rag_chain_english.invoke(query)
    
    # 한국어 또는 영어가 아닌 경우 에러 메시지 반환
    else:
        return "Unsupported language (Korean or English only)"

In [ ]:
# 한국어 쿼리에 대한 테스트 실행
query_ko = "테슬라 창업자는 누구인가요?"

output = run_route_rag_chain.invoke(query_ko)
print(output)

In [ ]:
# 영어 쿼리에 대한 테스트 실행
query_en = "Who is the founder of Tesla?"

output = run_route_rag_chain.invoke(query_en)
print(output)

---

##  **ReAct (Reasoning and Acting)**

- **ReAct** Agent는 Reasoning과 Acting을 결합한 가장 일반적인 에이전트 형태임

- 에이전트는 **행동-관찰-추론** 단계를 순환하며 작업을 수행함
    - 행동 (act): 모델이 특정 도구(Tool)를 호출
    - 관찰 (observe): 도구의 출력(Tool Message)를 모델에 다시 전달
    - 추론 (reason): 모델이 도구 출력을 바탕으로 다음 행동을 결정 (예: 또 다른 도구를 호출하거나 직접 응답을 생성)

- **도구 호출**(act)과 **결과 분석**(observe)을 통해 다음 **행동을 결정**(reason)하는 체계적인 프로세스를 가짐


---

### 1. **도구(tool) 정의하기** 

- **ReAct 도구**는 명확한 입출력 인터페이스를 통해 정의됨

- 각 도구는 **특정 기능**을 수행하는 독립적인 컴포넌트로 구현됨

- 도구의 **입력과 출력** 형식을 명확히 정의하여 에이전트와의 상호작용을 보장함

`(1) RAG 체인 생성 (한국어, 영어 구분)`

In [ ]:
# RAG 체인 생성 (메타데이터를 포함해서 답변 생성)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.chat_models import init_chat_model

# 질문 템플릿
template = """Answer the question based only on the following context.
Do not use any external information or knowledge. 
If the answer is not in the context, answer "I don't know".
- For proper nouns (names of people, places, organizations, etc.), provide both Korean and English names in the format: 한글명(English Name) or English Name(한글명)
- Example: 세종대왕(King Sejong), Microsoft(마이크로소프트), New York(뉴욕)
- Use the same language as the question.

[Context]
{context}

[Question] 
{question}

[Answer]
"""

# 프롬프트 생성
prompt = ChatPromptTemplate.from_template(template)

# LLM 모델 생성
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

# 문서 포맷터 함수
def format_docs_with_metadata(docs):
    formatted_docs = []
    for doc in docs:
        content = doc.page_content
        metadata = doc.metadata
        source = metadata.get('source', '출처 없음')
        formatted_docs.append(f"내용: {content}\n출처: {source}")
    return "\n\n".join(formatted_docs)


# RAG 체인 생성 (메타데이터 포함)
def create_rag_chain_with_metadata(vectorstore, top_k=2):
    """벡터 저장소에서 문서를 검색하여 메타데이터를 포함한 답변을 생성하는 RAG 체인 생성"""

    # 벡터 저장소에서 문서를 검색
    retriever = vectorstore.as_retriever(search_kwargs={'k': top_k})
    
    chain = RunnablePassthrough.assign(
        context=lambda x: format_docs_with_metadata(retriever.invoke(x["question"]))
    ) | RunnableParallel(
        context=lambda x: x["context"],
        answer=prompt | llm | StrOutputParser()
    )
    
    return chain

# 한국어 RAG 체인 생성
rag_chain_korean = create_rag_chain_with_metadata(db_korean, top_k=4)

# 한국어 RAG 체인 실행
response = rag_chain_korean.invoke({"question": "테슬라 창업자는 누구인가요?"})

pprint(response)

In [ ]:
# 영어 RAG 체인 생성
rag_chain_english = create_rag_chain_with_metadata(db_english, top_k=4)

# 영어 RAG 체인 실행
response = rag_chain_english.invoke({"question": "Who is the founder of Tesla?"})

pprint(response)

`(2) RAG 체인을 Tool 객체로 변환`

In [ ]:
# 한국어 RAG 도구 생성 (한국어 문서 벡터 저장소 사용)
rag_tool_korean = rag_chain_korean.as_tool(
    name="rag_korean_db",
    description="한국어 질문에 대한 리비안, 테슬라 관련 문서를 벡터 저장소에서 검색하고, 그 결과와 함께 답변을 생성합니다."
)

print(f"Tool 이름: {rag_tool_korean.name}")
print(f"Tool 설명: {rag_tool_korean.description}")
print(f"Tool 입력 파라미터: ")
pprint(rag_tool_korean.args)

In [ ]:
# 영어 RAG 도구 생성 (영어 문서 벡터 저장소 사용)
rag_tool_english = rag_chain_english.as_tool(
    name="rag_english_db",
    description="Retrieve and generate answers from the vector store for English questions related to Rivian and Tesla."
)

print(f"Tool 이름: {rag_tool_english.name}")
print(f"Tool 설명: {rag_tool_english.description}")
print(f"Tool 입력 파라미터: ")
pprint(rag_tool_english.args)

---

### 2. **도구(tool) 호출하기** 

- **bind_tools** 메서드로 LLM에 도구들을 연결하여 사용 가능하게 함

- **도구 호출 결과**는 ToolCall 객체를 통해 체계적으로 확인할 수 있음

In [ ]:
from langchain.chat_models import init_chat_model

# 도구 목록
tools = [rag_tool_korean, rag_tool_english]

# LLM 모델 
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

# 모델에 도구를 바인딩 (추가)
llm_with_tools = llm.bind_tools(tools=tools)

# 도구 사용하기 
query = "테슬라 창업자는 누구인가요?"
response = llm_with_tools.invoke(query)

pprint(response)

In [ ]:
# ToolCall 객체 확인
response.tool_calls

In [ ]:
# 영어 도구에 대한 질문
query_en = "Who is the founder of Tesla?"
response_en = llm_with_tools.invoke(query_en)

pprint(response_en)

In [ ]:
# ToolCall 객체 확인
response_en.tool_calls

In [ ]:
# 도구와 관련 없는 질문 테스트
query_test = "오늘 날씨는 어떤가요?"
response_test = llm_with_tools.invoke(query_test)

pprint(response_test)

In [ ]:
# ToolCall 객체 확인
response_test.tool_calls

---

### 3. **도구(tool) 실행하기** 

- **도구 호출 함수**는 AIMessage의 tool_calls를 실행하고 결과를 반환하는 헬퍼 함수로 구현

- **tool_map**을 통해 각 도구별 호출을 처리하며 invoke 메소드로 실행됨

- **최종 체인**은 llm_with_tools와 call_tools를 파이프라인으로 연결하여 구성됨

`(1) 도구 이름을 기준으로 매핑 정의`

In [ ]:
# 도구 맵 생성
tool_map = {
    "rag_korean_db": rag_tool_korean,
    "rag_english_db": rag_tool_english
}

# 도구 맵을 사용하여 도구 이름을 도구 객체로 변환 (도구 이름을 키로 사용)
tool_map["rag_korean_db"].invoke({"question": "테슬라 창업자는 누구인가요?"})

In [ ]:
# 도구 맵을 사용하여 도구 이름을 도구 객체로 변환 (영어)
tool_map["rag_english_db"].invoke({"question": "Who is the founder of Tesla?"})

`(2) 도구 호출 함수 정의`

In [ ]:
from langchain_core.messages import AIMessage

# 도구 호출 함수 정의
def call_tools(msg: AIMessage):
    """
    tool calling helper 함수: AIMessage에 있는 tool_calls를 실행하고 결과를 반환
    """
    tool_calls = msg.tool_calls.copy()
    for tool_call in tool_calls:
        tool_call["output"] = tool_map[tool_call["name"]].invoke(tool_call["args"])
    return tool_calls


# 도구 호출 함수를 사용하여 도구 호출 실행 
print("ToolCall 객체: ")
pprint(response.tool_calls[0])
print("-"*200)

tool_calls = call_tools(response)  # 도구 호출 실행 (AIMessage 객체를 입력으로 사용)
pprint(tool_calls)

`(3) 도구 호출 및 실행 체인 정의`

In [ ]:
# 도구 호출 체인 생성
search_tool_chain = llm_with_tools | call_tools

# 도구 호출 실행 (한국어 쿼리)
query = "테슬라 창업자는 누구인가요?"
search_response = search_tool_chain.invoke(query)

pprint(search_response)

In [ ]:
# 도구 호출 실행 (영어 쿼리)
query = "Who is the founder of Tesla?"
search_response = search_tool_chain.invoke(query)

pprint(search_response)

In [ ]:
# 도구 호출 실행 (한국어 쿼리) - 도구와 관련 없는 질문
query = "오늘 날씨는 어떤가요?"
search_response = search_tool_chain.invoke(query)

pprint(search_response)

---

### 4. **Agent** 

- **create_agent**는 LangChain v1.0의 표준 에이전트 생성 함수

- **LLM(대규모 언어 모델)** 을 의사결정 엔진으로 사용하여 작업을 수행하는 시스템

- 에이전트의 **계획-실행-관찰** 사이클을 자동으로 관리

- 에이전트의 행동을 **모니터링**하고 결과를 반환

- LangGraph를 기반으로 구축되어 **영속성, 스트리밍, Human-in-the-loop** 등의 기능을 자동 지원


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# 모델 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 도구 목록 생성 
tools = [rag_tool_korean, rag_tool_english]

# 에이전트 생성
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 사용자의 요청을 처리하는 AI Assistant입니다."
)

In [ ]:
# 에이전트 실행 (한국어 쿼리)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "테슬라 창업자는 누구인가요?"}]},
)

In [ ]:
# 에이전트 실행 결과 출력
pprint(response)

In [ ]:
# 에이전트 실행 (영어 쿼리)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Who is the founder of Tesla?"}]}
)

In [ ]:
# 에이전트 실행 결과 출력
pprint(response)

In [ ]:
# 에이전트 실행 (한국어 쿼리) - 도구와 관련 없는 질문
response = agent.invoke(
    {"messages": [{"role": "user", "content": "오늘 날씨는 어떤가요?"}]}
)

# 에이전트 실행 결과 출력
pprint(response)

---
# **[실습]**

- 언어 감지 및 번역 자동화 방식의 다국어 RAG 시스템을 구현합니다. (한국어, 영어, 중국어, 일본어 등)
- 이때, 사용자의 언어 감지 결과에 따라, 한국어와 다른 언어 간의 번역을 처리하는 도구를 별도로 구현합니다. 

- 언어 감지 결과에 따라 라우팅을 처리하고, 벡터 저장소는 한국어 DB만을 사용합니다. 

- 마지막으로, LangChain AgentExecutor 기반의 에이전트를 적용합니다. 

In [ ]:
from langchain_core.tools import tool
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langdetect import detect
import deepl
import time
from functools import wraps
from typing import Dict, Any, Optional
import hashlib

print("=" * 80)
print("[1단계] 언어 감지 도구 구현")
print("=" * 80)

# ------------------------------------------
# 1-1. 언어 감지 도구 정의
# ------------------------------------------
# 사용자 입력 텍스트가 어떤 언어인지 자동으로 판별하는 도구
# langdetect 라이브러리를 사용하여 언어 코드를 반환
# 예: 'ko' (한국어), 'en' (영어), 'zh-cn' (중국어), 'ja' (일본어)

@tool
def detect_language(text: str) -> str:
    """
    입력 텍스트의 언어를 감지하여 언어 코드를 반환합니다.
    
    Args:
        text: 언어를 감지할 텍스트
        
    Returns:
        str: 감지된 언어 코드 (예: 'ko', 'en', 'zh-cn', 'ja')
    """
    try:
        # langdetect 라이브러리로 언어 감지
        detected_lang = detect(text)
        return detected_lang
    except Exception as e:
        # 에러 발생 시 한국어로 가정
        print(f"[WARNING] 언어 감지 실패, 한국어로 가정: {e}")
        return "ko"

# 도구 테스트
print("\n[언어 감지 도구 테스트]")
test_texts = [
    "테슬라는 전기차를 만드는 회사입니다.",
    "Tesla is an electric vehicle company.",
    "テスラは電気自動車会社です。",
    "特斯拉是一家电动汽车公司。"
]

for text in test_texts:
    lang = detect_language.invoke(text)
    print(f"텍스트: {text[:30]}... -> 언어: {lang}")


print("\n" + "=" * 80)
print("[2단계] 번역 도구 구현 (캐싱 + 에러 핸들링 추가)")
print("=" * 80)

# ------------------------------------------
# 번역 캐시
# ------------------------------------------
# 번역 결과를 메모리에 캐싱하여 동일한 텍스트에 대한 반복 번역 방지

class TranslationCache:
    """
    번역 결과를 캐싱하여 API 호출 최소화 및 성능 향상
    
    학습 포인트:
    - 해시를 사용한 캐시 키 생성
    - 메모리 기반 딕셔너리 캐시
    - 캐시 히트율 추적
    """
    def __init__(self):
        self._cache: Dict[str, str] = {}
        self._hits = 0
        self._misses = 0
    
    def _make_key(self, text: str, target_lang: str) -> str:
        """텍스트와 목표 언어를 조합하여 캐시 키 생성"""
        # MD5 해시를 사용하여 고정 길이 키 생성
        key_string = f"{text}:{target_lang}"
        return hashlib.md5(key_string.encode()).hexdigest()
    
    def get(self, text: str, target_lang: str) -> Optional[str]:
        """캐시에서 번역 결과 조회"""
        key = self._make_key(text, target_lang)
        if key in self._cache:
            self._hits += 1
            return self._cache[key]
        self._misses += 1
        return None
    
    def set(self, text: str, target_lang: str, translation: str):
        """번역 결과를 캐시에 저장"""
        key = self._make_key(text, target_lang)
        self._cache[key] = translation
    
    def get_stats(self) -> Dict[str, Any]:
        """캐시 통계 반환"""
        total = self._hits + self._misses
        hit_rate = (self._hits / total * 100) if total > 0 else 0
        return {
            "hits": self._hits,
            "misses": self._misses,
            "total": total,
            "hit_rate": f"{hit_rate:.2f}%",
            "cache_size": len(self._cache)
        }

# 전역 번역 캐시 인스턴스
translation_cache = TranslationCache()


# ------------------------------------------
# 성능 측정 데코레이터
# ------------------------------------------
# 함수 실행 시간을 자동으로 측정하여 성능 분석에 활용

def measure_time(func):
    """
    함수 실행 시간을 측정하는 데코레이터
    
    학습 포인트:
    - 데코레이터 패턴으로 기능 확장
    - 실행 시간 측정으로 병목 지점 파악
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        elapsed = end_time - start_time
        print(f"[PERF] {func.__name__} 실행 시간: {elapsed:.3f}초")
        return result
    return wrapper


# ------------------------------------------
# 2-1. 한국어로 번역하는 도구
# ------------------------------------------

@tool
@measure_time
def translate_to_korean(text: str, source_lang: str = "auto") -> str:
    """
    입력 텍스트를 한국어로 번역합니다.
    
    Deep Dive 기능:
    - 번역 결과 캐싱으로 성능 향상
    - 실행 시간 자동 측정
    - 에러 핸들링 및 재시도 로직
    
    Args:
        text: 번역할 텍스트
        source_lang: 원본 언어 코드 (기본값: 'auto' - 자동 감지)
        
    Returns:
        str: 한국어로 번역된 텍스트
    """
    try:
        # 이미 한국어인 경우 원본 반환
        detected = detect(text)
        if detected == 'ko':
            return text
        
        # 캐시 확인
        cached_result = translation_cache.get(text, 'KO')
        if cached_result:
            print(f"[CACHE HIT] 캐시에서 번역 결과 반환")
            return cached_result
        
        # DeepL 번역기 객체 생성
        translator = deepl.Translator(os.getenv('DEEPL_API_KEY'))
        
        # 재시도 로직: 최대 3번 시도
        max_retries = 3
        for attempt in range(max_retries):
            try:
                result = translator.translate_text(text, target_lang='KO')
                translated_text = str(result)
                
                # 캐시에 저장
                translation_cache.set(text, 'KO', translated_text)
                
                return translated_text
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"[RETRY] 번역 시도 {attempt + 1} 실패, 재시도 중...")
                    time.sleep(1)  # 1초 대기 후 재시도
                else:
                    raise e
                    
    except Exception as e:
        # 번역 실패 시 원본 텍스트 반환 (fallback)
        print(f"[ERROR] 번역 오류: {e}")
        print(f"[FALLBACK] 원본 텍스트 반환")
        return text

# 도구 테스트
print("\n[한국어 번역 도구 테스트 - 캐싱 효과 확인]")
test_text = "Who founded Tesla?"
print(f"\n첫 번째 번역 (캐시 미스 예상):")
result1 = translate_to_korean.invoke(test_text)
print(f"결과: {result1}")

print(f"\n두 번째 번역 (캐시 히트 예상):")
result2 = translate_to_korean.invoke(test_text)
print(f"결과: {result2}")

print(f"\n캐시 통계:")
print(translation_cache.get_stats())


# ------------------------------------------
# 2-2. 한국어에서 다른 언어로 번역하는 도구
# ------------------------------------------

@tool
@measure_time
def translate_from_korean(text: str, target_lang: str) -> str:
    """
    한국어 텍스트를 지정된 언어로 번역합니다.
    
    Deep Dive 기능:
    - 캐싱 지원
    - 성능 측정
    - 에러 핸들링
    
    Args:
        text: 번역할 한국어 텍스트
        target_lang: 목표 언어 코드 (예: 'EN-US', 'JA', 'ZH')
        
    Returns:
        str: 목표 언어로 번역된 텍스트
    """
    try:
        # 목표 언어가 한국어인 경우 원본 반환
        if target_lang.upper() in ['KO', 'KOREAN']:
            return text
        
        # DeepL API의 언어 코드 형식으로 변환
        deepl_lang_map = {
            'en': 'EN-US',
            'ja': 'JA',
            'zh-cn': 'ZH',
            'zh-tw': 'ZH',
            'es': 'ES',
            'fr': 'FR',
            'de': 'DE'
        }
        
        target = deepl_lang_map.get(target_lang.lower(), target_lang.upper())
        
        # 캐시 확인
        cached_result = translation_cache.get(text, target)
        if cached_result:
            print(f"[CACHE HIT] 캐시에서 번역 결과 반환")
            return cached_result
        
        # DeepL 번역기 객체 생성
        translator = deepl.Translator(os.getenv('DEEPL_API_KEY'))
        
        # 번역 수행 (재시도 로직 포함)
        max_retries = 3
        for attempt in range(max_retries):
            try:
                result = translator.translate_text(text, target_lang=target)
                translated_text = str(result)
                
                # 캐시에 저장
                translation_cache.set(text, target, translated_text)
                
                return translated_text
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"[RETRY] 번역 시도 {attempt + 1} 실패, 재시도 중...")
                    time.sleep(1)
                else:
                    raise e
                    
    except Exception as e:
        print(f"[ERROR] 번역 오류: {e}")
        print(f"[FALLBACK] 원본 텍스트 반환")
        return text


print("\n" + "=" * 80)
print("[3단계] RAG 검색 도구 구현")
print("=" * 80)

# ------------------------------------------
# 3-1. 한국어 벡터 저장소 기반 RAG 도구
# ------------------------------------------

@tool
@measure_time
def search_korean_db(query: str) -> dict:
    """
    한국어 쿼리로 벡터 저장소를 검색하고 답변을 생성합니다.
    
    Deep Dive 기능:
    - 실행 시간 측정
    - 검색 품질 메트릭 수집
    
    Args:
        query: 검색할 한국어 쿼리
        
    Returns:
        dict: 검색 컨텍스트, 답변, 메타데이터 포함
    """
    try:
        # 벡터 저장소에서 관련 문서 검색
        retriever = db_korean.as_retriever(search_kwargs={'k': 4})
        
        # 검색 시간 측정
        search_start = time.time()
        docs = retriever.invoke(query)
        search_time = time.time() - search_start
        
        context = format_docs_with_metadata(docs)
        
        # RAG 체인을 통해 답변 생성
        rag_chain = create_rag_chain_with_metadata(db_korean, top_k=4)
        
        # 답변 생성 시간 측정
        generate_start = time.time()
        result = rag_chain.invoke({"question": query})
        generate_time = time.time() - generate_start
        
        # 성능 메트릭 추가
        result["metrics"] = {
            "search_time": f"{search_time:.3f}s",
            "generate_time": f"{generate_time:.3f}s",
            "docs_retrieved": len(docs)
        }
        
        return result
        
    except Exception as e:
        print(f"[ERROR] RAG 검색 오류: {e}")
        return {
            "context": "",
            "answer": "검색 중 오류가 발생했습니다.",
            "metrics": {"error": str(e)}
        }


print("\n" + "=" * 80)
print("[4단계] 통합 다국어 RAG 워크플로우 도구")
print("=" * 80)

# ------------------------------------------
# 성능 추적을 위한 메트릭 수집기
# ------------------------------------------

class PerformanceTracker:
    """
    시스템 전체의 성능 메트릭을 수집하고 분석
    
    학습 포인트:
    - 메트릭 기반 성능 분석
    - 병목 지점 식별
    - 시스템 개선 방향 도출
    """
    def __init__(self):
        self.queries = []
    
    def log_query(self, query_data: Dict[str, Any]):
        """쿼리 실행 정보 기록"""
        self.queries.append(query_data)
    
    def get_summary(self) -> Dict[str, Any]:
        """전체 성능 요약 통계"""
        if not self.queries:
            return {"message": "수집된 데이터 없음"}
        
        total_queries = len(self.queries)
        languages = [q.get('language') for q in self.queries]
        avg_time = sum(q.get('total_time', 0) for q in self.queries) / total_queries
        
        return {
            "total_queries": total_queries,
            "languages": dict((lang, languages.count(lang)) for lang in set(languages)),
            "avg_response_time": f"{avg_time:.3f}s",
            "cache_stats": translation_cache.get_stats()
        }

# 전역 성능 추적기
perf_tracker = PerformanceTracker()


@tool
def multilingual_rag_search(query: str) -> str:
    """
    다국어 쿼리를 처리하여 RAG 검색을 수행하고 원래 언어로 답변을 반환합니다.
    
    Deep Dive 기능:
    - 전체 워크플로우 성능 측정
    - 각 단계별 시간 추적
    - 성능 메트릭 수집
    
    Args:
        query: 검색 쿼리 (모든 언어 지원)
        
    Returns:
        str: 원래 언어로 된 답변
    """
    workflow_start = time.time()
    metrics = {}
    
    print(f"\n[워크플로우 시작] 원본 쿼리: {query}")
    
    try:
        # Step 1: 언어 감지
        step_start = time.time()
        detected_lang = detect_language.invoke(query)
        metrics['language_detection'] = time.time() - step_start
        print(f"[Step 1] 감지된 언어: {detected_lang} ({metrics['language_detection']:.3f}s)")
        
        # Step 2: 한국어로 번역
        step_start = time.time()
        if detected_lang != 'ko':
            korean_query = translate_to_korean.invoke(query)
            metrics['translation_to_korean'] = time.time() - step_start
            print(f"[Step 2] 한국어 번역: {korean_query} ({metrics['translation_to_korean']:.3f}s)")
        else:
            korean_query = query
            metrics['translation_to_korean'] = 0
            print(f"[Step 2] 이미 한국어입니다")
        
        # Step 3: RAG 검색
        step_start = time.time()
        search_result = search_korean_db.invoke(korean_query)
        metrics['rag_search'] = time.time() - step_start
        korean_answer = search_result['answer']
        print(f"[Step 3] RAG 검색 완료 ({metrics['rag_search']:.3f}s)")
        
        # Step 4: 원래 언어로 번역
        step_start = time.time()
        if detected_lang != 'ko':
            final_answer = translate_from_korean.invoke({
                'text': korean_answer,
                'target_lang': detected_lang
            })
            metrics['translation_from_korean'] = time.time() - step_start
            print(f"[Step 4] 최종 번역 완료 ({metrics['translation_from_korean']:.3f}s)")
        else:
            final_answer = korean_answer
            metrics['translation_from_korean'] = 0
            print(f"[Step 4] 번역 불필요")
        
        # 전체 실행 시간
        total_time = time.time() - workflow_start
        metrics['total_time'] = total_time
        
        # 성능 추적기에 기록
        perf_tracker.log_query({
            'query': query,
            'language': detected_lang,
            'total_time': total_time,
            'metrics': metrics
        })
        
        print(f"\n[성능 요약] 전체 실행 시간: {total_time:.3f}s")
        for step, duration in metrics.items():
            if step != 'total_time':
                percentage = (duration / total_time * 100) if total_time > 0 else 0
                print(f"  - {step}: {duration:.3f}s ({percentage:.1f}%)")
        
        return final_answer
        
    except Exception as e:
        print(f"[ERROR] 워크플로우 오류: {e}")
        return f"처리 중 오류가 발생했습니다: {str(e)}"


print("\n" + "=" * 80)
print("[5단계] Agent 기반 자동화 시스템 구축")
print("=" * 80)

# 에이전트가 사용할 도구 목록
agent_tools = [
    detect_language,
    translate_to_korean,
    translate_from_korean,
    search_korean_db,
    multilingual_rag_search
]

# LLM 모델 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 에이전트 시스템 프롬프트
system_prompt = """당신은 다국어 RAG 검색 시스템의 AI 어시스턴트입니다.

주요 역할:
1. 사용자의 질문 언어를 자동으로 감지합니다.
2. 한국어 벡터 데이터베이스를 활용하여 정보를 검색합니다.
3. 답변을 사용자가 사용한 언어로 제공합니다.

사용 가능한 도구:
- detect_language: 텍스트의 언어를 감지
- translate_to_korean: 다른 언어를 한국어로 번역
- search_korean_db: 한국어 벡터 DB 검색
- translate_from_korean: 한국어를 다른 언어로 번역
- multilingual_rag_search: 전체 프로세스를 한 번에 처리 (권장)

가장 효율적인 방법:
- 사용자 질문에 대해 multilingual_rag_search 도구를 사용하세요.
- 이 도구는 언어 감지, 번역, 검색을 자동으로 처리하며 성능 메트릭도 제공합니다.

답변 시 유의사항:
- 검색 결과에 기반한 정확한 정보를 제공하세요.
- 고유명사는 한글과 영문을 함께 표기하세요.
- 출처 정보가 있다면 함께 제공하세요.
"""

# 에이전트 생성
multilingual_agent = create_agent(
    model=llm,
    tools=agent_tools,
    system_prompt=system_prompt
)

print("\n[에이전트 생성 완료]")


print("\n" + "=" * 80)
print("[6단계] 종합 테스트 및 성능 비교")
print("=" * 80)

# ------------------------------------------
# 다양한 시나리오 테스트 및 비교
# ------------------------------------------

test_cases = [
    {"lang": "한국어", "query": "테슬라의 창업자는 누구인가요?"},
    {"lang": "영어", "query": "Who is the CEO of Rivian?"},
    {"lang": "일본어", "query": "リビアンはいつ設立されましたか？"},
    {"lang": "한국어", "query": "테슬라와 리비안의 차이점을 설명해주세요."}
]

print("\n[종합 테스트 실행]")
for i, test_case in enumerate(test_cases, 1):
    print(f"\n{'='*60}")
    print(f"[테스트 {i}] {test_case['lang']} 질문")
    print(f"{'='*60}")
    print(f"질문: {test_case['query']}")
    
    response = multilingual_agent.invoke(
        {"messages": [{"role": "user", "content": test_case['query']}]}
    )
    
    print(f"\n답변: {response['messages'][-1].content}")
    print(f"{'='*60}")


print("\n" + "=" * 80)
print("[7단계] - 성능 분석")
print("=" * 80)

# ------------------------------------------
# 전체 시스템 성능 분석
# ------------------------------------------

print("\n[전체 성능 요약]")
performance_summary = perf_tracker.get_summary()
print("\n1. 실행 통계:")
for key, value in performance_summary.items():
    print(f"   - {key}: {value}")

print("\n2. 번역 캐시 효율성:")
cache_stats = translation_cache.get_stats()
for key, value in cache_stats.items():
    print(f"   - {key}: {value}")


[1단계] 언어 감지 도구 구현

[언어 감지 도구 테스트]
텍스트: 테슬라는 전기차를 만드는 회사입니다.... -> 언어: ko
텍스트: Tesla is an electric vehicle c... -> 언어: en
텍스트: テスラは電気自動車会社です。... -> 언어: ja
텍스트: 特斯拉是一家电动汽车公司。... -> 언어: zh-cn

[2단계] 번역 도구 구현 (캐싱 + 에러 핸들링 추가)

[한국어 번역 도구 테스트 - 캐싱 효과 확인]

첫 번째 번역 (캐시 미스 예상):
[PERF] translate_to_korean 실행 시간: 1.512초
결과: 테슬라를 설립한 사람은 누구인가요?

두 번째 번역 (캐시 히트 예상):
[CACHE HIT] 캐시에서 번역 결과 반환
[PERF] translate_to_korean 실행 시간: 0.002초
결과: 테슬라를 설립한 사람은 누구인가요?

캐시 통계:
{'hits': 1, 'misses': 1, 'total': 2, 'hit_rate': '50.00%', 'cache_size': 1}

[3단계] RAG 검색 도구 구현

[4단계] 통합 다국어 RAG 워크플로우 도구

[5단계] Agent 기반 자동화 시스템 구축

[에이전트 생성 완료]

[6단계] 종합 테스트 및 성능 비교

[종합 테스트 실행]

[테스트 1] 한국어 질문
질문: 테슬라의 창업자는 누구인가요?

[워크플로우 시작] 원본 쿼리: 테슬라의 창업자는 누구인가요?
[Step 1] 감지된 언어: ko (0.001s)
[Step 2] 이미 한국어입니다
[PERF] search_korean_db 실행 시간: 1.938초
[Step 3] RAG 검색 완료 (1.939s)
[Step 4] 번역 불필요

[성능 요약] 전체 실행 시간: 1.941s
  - language_detection: 0.001s (0.1%)
  - translation_to_korean: 0.000s (0.0%)
  - rag_search: 1.939s (99.9%)
  - t